## **EXACT ResNet-18 Division Strategy**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torchvision.models as models
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import random
import itertools
import math
import os
import time
import scipy.io
from scipy.io.matlab.mio5_params import mat_struct

def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

In [ ]:
# Strain Mappings
WT_STRAINS = [0, 1, 2, 3, 4]      # 5 strains, class 0
NWT_STRAINS = [5, 8, 9, 12, 17]       # 5 strains, class 1
ORDER = WT_STRAINS + NWT_STRAINS   # total_strains = 10

print(f"WT strains: {WT_STRAINS}")
print(f"NWT strains: {NWT_STRAINS}")
print(f"Total order: {ORDER}")

# Data Loading Functions
def _deep(o):
    if isinstance(o, mat_struct):
        return {f: _deep(getattr(o, f)) for f in o._fieldnames}
    if isinstance(o, np.ndarray):
        if o.dtype == object:
            if o.ndim == 0: return _deep(o.item())
            return [_deep(x) for x in o.ravel()]
        if o.dtype.names is not None:
            if o.shape == ():  # scalar structured
                return {n: _deep(o[idx][n]) for n in o.dtype.names}
            return [{n: _deep(o[idx][n]) for n in o.dtype.names}
                    for idx in np.ndindex(o.shape)]
        if o.ndim == 0: return o.item()
        return o
    if isinstance(o, np.void) and o.dtype.names is not None:
        return {n: _deep(o[n]) for n in o.dtype.names}
    return o

def _load_var_v5(mat_path, varname):
    part = scipy.io.loadmat(
        mat_path,
        variable_names=[varname],
        squeeze_me=False,
        struct_as_record=True,
        chars_as_strings=True,
    )
    if varname not in part:
        raise KeyError(f"{varname} not found in {mat_path}")
    return _deep(part[varname])

def _coerce_60x60(x):
    a = np.asarray(x, dtype=np.float32)
    a = np.squeeze(a)
    if a.shape == (60, 60):
        return a
    if a.ndim == 3:
        for ax in range(3):
            if a.shape[ax] == 1:
                b = np.squeeze(a, axis=ax)
                if b.shape == (60, 60):
                    return b.astype(np.float32)
    raise ValueError(f"Unexpected image shape {a.shape}")

def _find_image(o):
    if isinstance(o, dict):
        if "image" in o: return _coerce_60x60(o["image"])
        for v in o.values():
            try: return _find_image(v)
            except Exception: pass
        raise KeyError("No 'image' in dict")
    if isinstance(o, (list, tuple)):
        for v in o:
            try: return _find_image(v)
            except Exception: pass
        raise KeyError("No 'image' in list/tuple")
    if isinstance(o, np.ndarray):
        if o.dtype == object:
            for v in o.ravel():
                try: return _find_image(v)
                except Exception: pass
            raise KeyError("No 'image' in object array")
        if o.dtype.names is not None:
            for n in o.dtype.names:
                try: return _find_image(o[n])
                except Exception: pass
            raise KeyError("No 'image' in structured array")
        return _coerce_60x60(o)
    if isinstance(o, np.void) and o.dtype.names is not None:
        for n in o.dtype.names:
            try: return _find_image(o[n])
            except Exception: pass
        raise KeyError("No 'image' in np.void")
    raise TypeError(f"Unsupported item type {type(o)}")

def f_read_bacteria_WT_NWT_py(mat_path: str, n_cells_perS: int):
    """Loads WT vs NWT images robustly"""
    TARGET_ORDER = ORDER
    usable_order = []
    loaded_cells_per_strain = {}

    for s in TARGET_ORDER:
        name = f'class_{s}'
        try:
            cells = _load_var_v5(mat_path, name)
            if isinstance(cells, np.ndarray) and cells.dtype == object:
                cells = [_deep(x) for x in cells.ravel()]
            elif not isinstance(cells, list):
                cells = [cells]
            if len(cells) >= n_cells_perS:
                usable_order.append(s)
                loaded_cells_per_strain[s] = cells
            else:
                raise ValueError(f"{name} has only {len(cells)} items (< {n_cells_perS})")
        except Exception as e:
            print(f" Skipping {name}: {e}")
            continue

    if not usable_order:
        raise RuntimeError("No expected class_* variables could be loaded from this file.")

    S = len(usable_order)
    H = W = 60
    N = S * n_cells_perS
    X = np.empty((H, W, N), dtype=np.float32)
    strain_id = np.empty((N, 1), dtype=np.int32)
    class_id  = np.empty((N, 1), dtype=np.int32)

    ptr = 0
    for j, strain in enumerate(usable_order, start=1):
        cells = loaded_cells_per_strain[strain]
        for i in range(n_cells_perS):
            X[:, :, ptr + i] = _find_image(cells[i])
        strain_id[ptr:ptr+n_cells_perS, 0] = j
        class_id[ptr:ptr+n_cells_perS, 0]  = (0 if strain in WT_STRAINS else 1)
        ptr += n_cells_perS

    strain_ids_0 = np.asarray(usable_order, dtype=np.int32)
    strain_ids   = np.arange(1, S + 1, dtype=np.int32)
    return X, strain_ids_0, strain_ids, strain_id, class_id

In [ ]:
# Load Data
X_tr, s0_tr, s_tr, sid_tr, cid_tr = f_read_bacteria_WT_NWT_py(
    '/home/dineth/bacteria_QPM/data/bacteria_train_full_data.mat',
    n_cells_perS=11000
)

X_te, s0_te, s_te, sid_te, cid_te = f_read_bacteria_WT_NWT_py(
    '/home/dineth/bacteria_QPM/data/bacteria_test_full_data.mat',
    n_cells_perS=1300
)


print("\n TRAIN")
print("X_tr shape:", X_tr.shape)
print("strain_ids_0:", s0_tr)
print("strain_ids:", s_tr)
print("strain_id shape:", sid_tr.shape, "| unique:", np.unique(sid_tr))
print("class_id shape:", cid_tr.shape, "| unique:", np.unique(cid_tr))

print("\n TEST ")
print("X_te shape:", X_te.shape)
print("strain_ids_0:", s0_te)
print("strain_ids:", s_te)
print("strain_id shape:", sid_te.shape, "| unique:", np.unique(sid_te))
print("class_id shape:", cid_te.shape, "| unique:", np.unique(cid_te))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# Modified ResNet18EmbeddingExtractor with projection head (512→512→512)
class ResNet18EmbeddingExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # Load ResNet-18 backbone
        self.backbone = models.resnet18(pretrained=False)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Remove final classification layer
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-1])
        
        # SSL projection head (only used during pre-training)
        self.projection_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 512)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.projection_head(features)
        return embeddings
    
    def get_backbone_features(self, x):
        """Get embeddings using projection head"""
        features = self.backbone(x)
        embeddings = self.projection_head(features)
        return embeddings

# MAE Architecture (adapted from TV4)
class MaskedAutoencoder(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.embedding_dim = embedding_dim
        
        # Encoder: ResNet-18 backbone with projection head
        self.encoder = ResNet18EmbeddingExtractor()
        
        # Decoder: Updated for 512-dim embeddings
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, 512 * 2 * 2),
            nn.ReLU(),
            nn.Unflatten(1, (512, 2, 2)),
            # Upsample to 60x60 through multiple transposed convolutions
            nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 2x2 -> 4x4
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 4x4 -> 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 8x8 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),    # 16x16 -> 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, 2, 1),    # 32x32 -> 64x64
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, 1, 1),     # 64x64 -> 60x60
            nn.Sigmoid()
        )
    
    def forward(self, x, mask):
        # Apply mask to input
        x_masked = x * mask
        
        # Get embeddings
        embeddings = self.encoder(x_masked)
        
        # Reconstruct original image
        reconstruction = self.decoder(embeddings)
        
        # Ensure output matches input size
        if reconstruction.shape[-2:] != x.shape[-2:]:
            reconstruction = F.interpolate(reconstruction, size=x.shape[-2:], mode='bilinear', align_corners=False)
        
        return reconstruction, embeddings

# Updated Classification Head for 512 dimensions
class ClassificationHead(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=2):
        super().__init__()
        # Two-layer classifier: 512 → 256 → 2
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

In [ ]:
def get_dynamic_mask(x, epoch, total_epochs, base_ratio=0.3):
    """Generate dynamic masking patterns (from TV4)"""
    # Sinusoidal variation
    sinusoidal_ratio = base_ratio + 0.1 * np.sin(2 * np.pi * epoch / total_epochs)
    
    # Create random mask
    mask = torch.rand_like(x) > sinusoidal_ratio
    
    # Ensure some features are always masked
    min_mask_ratio = 0.1
    if mask.float().mean() > (1 - min_mask_ratio):
        mask = torch.rand_like(x) > (1 - min_mask_ratio)
    
    return mask.float()

def prepare_data_for_mae(X, batch_size=64, shuffle=True):
    """Convert numpy arrays to PyTorch DataLoader for MAE training"""
    # Reshape X from (60, 60, N) to (N, 1, 60, 60) for PyTorch
    X_reshaped = np.transpose(X, (2, 0, 1))  # (N, 60, 60)
    X_reshaped = np.expand_dims(X_reshaped, axis=1)  # (N, 1, 60, 60)
    
    X_tensor = torch.FloatTensor(X_reshaped)
    dataset = TensorDataset(X_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

def train_self_supervised_mae(X_all, epochs=50, lr=1e-3, batch_size=64, seed=123):
    """Train MAE with dynamic masking (adapted from TV4)"""
    set_seed(seed)
    mae = MaskedAutoencoder().to(DEVICE)
    optimizer = optim.AdamW(mae.parameters(), lr=lr, weight_decay=1e-4)
    
    # Warmup scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10)
    
    dataloader = prepare_data_for_mae(X_all, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        mae.train()
        total_loss = 0
        total_n = 0
        
        for batch_idx, (data,) in enumerate(dataloader):
            data = data.to(DEVICE)
            
            # Dynamic masking
            mask = get_dynamic_mask(data, epoch, epochs)
            
            # Forward pass
            reconstruction, embeddings = mae(data, mask)
            
            # Multi-component loss with proper dimension handling
            # 1. Reconstruction loss (only on masked regions)
            mse_loss = F.mse_loss(reconstruction * (1 - mask), data * (1 - mask))
            
            # 2. Embedding regularization
            embedding_reg = 0.01 * torch.mean(embeddings ** 2)
            
            # 3. Contrastive loss for diverse embeddings
            if embeddings.size(0) > 1:
                embeddings_norm = F.normalize(embeddings, p=2, dim=1)
                similarity = torch.mm(embeddings_norm, embeddings_norm.t())
                contrastive_loss = 0.001 * torch.mean(similarity ** 2)
            else:
                contrastive_loss = 0.0
            
            total_loss_component = mse_loss + embedding_reg + contrastive_loss
            
            optimizer.zero_grad()
            total_loss_component.backward()
            torch.nn.utils.clip_grad_norm_(mae.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += mse_loss.item() * data.size(0)
            total_n += data.size(0)
        
        scheduler.step()
        
        if epoch % 10 == 0 or epoch == 1 or epoch == epochs:
            current_mask_ratio = 1.0 - mask.mean().item()
            print(f"[MAE] Epoch {epoch}/{epochs}  Loss={total_loss/max(1,total_n):.6f}  mask_ratio={current_mask_ratio:.3f}")
    
    return mae

@torch.no_grad()
def extract_embeddings(mae, X, batch_size=512):
    """Extract embeddings using projection head (512-D)"""
    mae.eval()
    dataloader = prepare_data_for_mae(X, batch_size=batch_size, shuffle=False)
    
    embeddings = []
    for (data,) in dataloader:
        data = data.to(DEVICE)
        # Extract embeddings using projection head
        embeddings_batch = mae.encoder(data)  # This includes the projection head
        embeddings.append(embeddings_batch.cpu().numpy())
    
    return np.concatenate(embeddings, axis=0)

In [ ]:
def train_classification_head(
    embeddings, labels, test_embeddings, test_labels,
    epochs=30, lr=1e-3, batch_size=1024, label_type="True"
):  
    
    torch.manual_seed(123)
    torch.cuda.manual_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = ClassificationHead().to(device)
    
    # Fixed: label_smoothing 0.05 instead of 0.150
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    
    # Fixed: Separate weight decay for different param types
    decay_params = []
    no_decay_params = []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim == 1 or n.endswith('bias'):
            no_decay_params.append(p)
        else:
            decay_params.append(p)
    
    # Fixed: weight_decay 1e-5 instead of 1e-3, and only on decay_params
    optimizer = optim.AdamW([
        {'params': decay_params, 'weight_decay': 1e-5},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ], lr=lr)
    
    # Added: Cosine scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Move all data to GPU once
    Xtr = torch.as_tensor(embeddings, dtype=torch.float32, device=device)
    ytr = torch.as_tensor(labels, dtype=torch.long, device=device)
    Xte = torch.as_tensor(test_embeddings, dtype=torch.float32, device=device)
    yte = torch.as_tensor(test_labels, dtype=torch.long, device=device)

    # Lightweight DataLoader
    train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True, num_workers=0)

    best_acc = 0.0
    best_epoch = 0

    print(f"\n[{label_type} Labels] Training Classification Head:")
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for xb, yb in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)

        # Added: Step scheduler
        scheduler.step()

        # Evaluate after epoch
        model.eval()
        with torch.inference_mode():
            train_preds = model(Xtr).argmax(1)
            train_acc = (train_preds == ytr).float().mean().item() * 100.0

            test_preds = model(Xte).argmax(1)
            test_acc = (test_preds == yte).float().mean().item() * 100.0

        avg_loss = total_loss / len(Xtr)
        print(f"    Epoch {epoch:2d}: Loss = {avg_loss:.4f}, Train Acc = {train_acc:.2f}%, Test Acc = {test_acc:.2f}%")

        if test_acc > best_acc:
            best_acc = test_acc
            best_epoch = epoch

    print(f"    [BEST] Epoch {best_epoch}: Test Acc = {best_acc:.2f}%")
    return best_acc / 100.0

In [ ]:
# # Self-Supervised Pre-training with MAE
# print("\n[Self-Supervised Pre-training on ALL data]")
# # Combine train and test data for pre-training
# X_all = np.concatenate([X_tr, X_te], axis=2)
# print(f"Combined data shape: {X_all.shape}")

# # Train MAE (using TV4's successful approach)
# mae = train_self_supervised_mae(
#     X_all, 
#     epochs=50,        # Same as TV4
#     lr=1e-3, 
#     batch_size=64,
#     seed=123
# )

# print("\n[Extracting embeddings]")
# train_embeddings = extract_embeddings(mae, X_tr, batch_size=512)
# test_embeddings = extract_embeddings(mae, X_te, batch_size=512)

# print(f"Train embeddings shape: {train_embeddings.shape}")
# print(f"Test embeddings shape: {test_embeddings.shape}")

# # Save embeddings
# os.makedirs('/home/dineth/bacteria_QPM/WT_NWT/Pre-train/embeddings', exist_ok=True)
# np.save('/home/dineth/bacteria_QPM/WT_NWT/Pre-train/embeddings/train_embeddings_mae_512.npy', train_embeddings)
# np.save('/home/dineth/bacteria_QPM/WT_NWT/Pre-train/embeddings/test_embeddings_mae_512.npy', test_embeddings)

# print("Embeddings saved!")

In [ ]:
train_embeddings = np.load("/home/dineth/bacteria_QPM/WT_NWT/Pre-train/embeddings/train_embeddings_mae_512.npy")
test_embeddings = np.load("/home/dineth/bacteria_QPM/WT_NWT/Pre-train/embeddings/test_embeddings_mae_512.npy")

In [ ]:
def print_permutation_mapping(perm_idx, strain_to_label):
    print(f"\nPermutation {perm_idx}: strain -> class (TRUE -> PERMUTED)")
    for sid in ORDER:
        true_class = 0 if sid in WT_STRAINS else 1
        perm_class = strain_to_label[sid]
        print(f"Strain {sid:2d} | {'WT' if true_class == 0 else 'NWT':<3} -> {'WT' if perm_class == 0 else 'NWT'}")

In [ ]:
#Permutation Test Setup
total_strains = 10
wt_strains = 5  # number of 0s
nwt_strains = 5  # number of 1s

unique_permutations_count = math.factorial(total_strains) // (math.factorial(wt_strains) * math.factorial(nwt_strains))
print(f"\nNumber of unique permutations: {unique_permutations_count}")

def generate_all_unique_permutations():
    base_pattern = [0] * wt_strains + [1] * nwt_strains
    unique_patterns = list(set(itertools.permutations(base_pattern)))
    original_pattern = tuple(base_pattern)
    
    if original_pattern not in unique_patterns:
        unique_patterns[0] = original_pattern
        unique_patterns = list(set(unique_patterns))
    unique_patterns.sort()
    print(f"Generated {len(unique_patterns)} unique patterns")
    return unique_patterns, original_pattern

# FIXED: Unpack the tuple properly
all_unique_patterns, original_pattern = generate_all_unique_permutations()
N_PERMUTATIONS = len(all_unique_patterns) 

print(f"\nRunning {N_PERMUTATIONS} unique permutations...")

In [ ]:
import time

start_time = time.time()
null_accuracies = []  # Start empty - no explicit true accuracy addition

for perm_idx, perm_pattern in enumerate(all_unique_patterns, 1):
    print(f"\n{'='*60}")
    print(f"PERMUTATION {perm_idx}/{N_PERMUTATIONS}")
    print(f"{'='*60}")
    
    # Create strain-to-label mapping using the current unique pattern (FIXED)
    strain_to_label = {strain: perm_pattern[i] for i, strain in enumerate(ORDER)}
    
    # Apply the mapping to both splits (FIXED)
    y_train_permuted = np.array([strain_to_label[s] for s in s0_tr], dtype=int)
    y_test_permuted = np.array([strain_to_label[s] for s in s0_te], dtype=int)
    
    # Map to actual sample labels
    y_train_permuted_samples = np.array([strain_to_label[s0_tr[i]] for i in sid_tr.flatten() - 1], dtype=int)
    y_test_permuted_samples = np.array([strain_to_label[s0_te[i]] for i in sid_te.flatten() - 1], dtype=int)
    
    # Print permutation mapping
    print_permutation_mapping(perm_idx, strain_to_label)
    
    # Train & evaluate with permuted labels
    perm_accuracy = train_classification_head(
        train_embeddings, 
        y_train_permuted_samples,
        test_embeddings,
        y_test_permuted_samples,
        epochs=5, 
        lr=0.01, 
        label_type=f"Permutation {perm_idx}"
    )
    null_accuracies.append(perm_accuracy)
    
    print(f"\nPermutation {perm_idx} final accuracy: {perm_accuracy:.4f}")
    
    if perm_idx % 10 == 0:  
        print(f"\n[PROGRESS] Completed {perm_idx}/{N_PERMUTATIONS} permutations")
        print(f"Current mean accuracy: {np.mean(null_accuracies):.4f}")
        print(f"Current std accuracy: {np.std(null_accuracies):.4f}")

# End timing
end_time = time.time()
total_time = end_time - start_time
avg_perm_time = total_time / N_PERMUTATIONS
print(total_time)


In [ ]:
import pandas as pd

# Create DataFrame with permutation patterns and accuracies
results_data = []
for perm_idx, perm_pattern in enumerate(all_unique_patterns, 1):
    # Convert permutation pattern tuple to string format
    perm_pattern_str = ','.join(map(str, perm_pattern))
    accuracy = null_accuracies[perm_idx - 1]  # perm_idx is 1-indexed, list is 0-indexed
    
    results_data.append({
        'permutation_index': perm_idx,
        'permutation_pattern': perm_pattern_str,
        'accuracy': accuracy
    })

# Create DataFrame
results_df = pd.DataFrame(results_data)

# Get notebook name (without extension) for CSV filename
notebook_name = 'WT_NWTV5'  # or extract from __file__ if available
csv_filename = f'{notebook_name}.csv'

# Save to CSV
csv_path = f'/home/dineth/bacteria_QPM/WT_NWT/Pre-train/{csv_filename}'
results_df.to_csv(csv_path, index=False)

print(f"\nResults saved to: {csv_path}")
print(f"Total rows: {len(results_df)}")
print(f"\nFirst few rows:")
print(results_df.head())

In [ ]:
null_accuracies = np.array(null_accuracies)  # All 252 accuracies (including true one) is
true_perm = tuple([0] * wt_strains + [1] * nwt_strains)
true_perm_idx = all_unique_patterns.index(true_perm)
true_accuracy = null_accuracies[true_perm_idx]

# Calculate p-value: how many permutations (including true) are >= true accuracy
p_value = np.mean(null_accuracies >= true_accuracy)

print(f"\n[RESULTS]")
print(f"True labels accuracy:    {true_accuracy:.4f}")
print(f"All accuracies mean ± std:  {null_accuracies.mean():.4f} ± {null_accuracies.std():.4f}")
print(f"P-value (>= observed):   {p_value:.6f}")
print(f"95th percentile:         {np.percentile(null_accuracies, 95):.4f}")
print(f"99th percentile:         {np.percentile(null_accuracies, 99):.4f}")
print(f"Max accuracy:            {null_accuracies.max():.4f}")

# Visualization (All 252 accuracies in histogram, including true one)
plt.figure(figsize=(10, 6))
plt.hist(null_accuracies * 100, bins=30, color='navy', alpha=0.7, edgecolor='black', 
         label=f'All Results ({N_PERMUTATIONS} permutations including true)')

# Observed performance line (true accuracy is now part of the histogram)
plt.axvline(true_accuracy * 100, color='red', linewidth=3, label='True Performance')

# Chance baseline for binary classification = 50%
plt.axvline(50, color='k', linestyle=':', linewidth=2, label='Chance (50%)')

plt.xlabel("Accuracy [%] (on the test set)")
plt.ylabel("Frequency")
plt.title("Self-Supervised Permutation Test Results: WT vs NWT Classification\n(ResNet-18 MAE + Classification Head)")

# Add text annotation with observed accuracy and permutation p-value
ymax = max(np.histogram(null_accuracies * 100, bins=30)[0])
plt.text(true_accuracy * 100 + 2, ymax * 0.7,
         f"Observed = {true_accuracy * 100:.2f}%\nP-value = {p_value:.4f}",
         fontsize=12, bbox=dict(facecolor='yellow', alpha=0.7))

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Create a matrix where each row is a permutation pattern
perm_matrix = np.array([list(perm) for perm in all_unique_patterns])

# Create the color plot
plt.figure(figsize=(14, 20))
im = plt.imshow(perm_matrix, cmap='RdYlBu_r', aspect='auto', interpolation='nearest')

# Set labels
plt.xlabel('Strain (in ORDER)', fontsize=12)
plt.ylabel('Permutation Index', fontsize=12)
plt.title('Permutation Patterns: 0s (WT) and 1s (NWT)\n120 Unique Permutations', fontsize=14)

# Set x-axis ticks to show strain IDs
plt.xticks(range(len(ORDER)), ORDER, fontsize=10)
plt.yticks(range(0, len(all_unique_patterns), 10), fontsize=8)

# Add colorbar
cbar = plt.colorbar(im, ticks=[0, 1])
cbar.set_label('Class (0=WT, 1=NWT)', rotation=270, labelpad=20)

# Add grid for better readability
plt.grid(True, alpha=0.3, color='white', linewidth=0.5)

plt.tight_layout()
plt.show()

# Also print some statistics
print(f"\nTotal permutations: {len(all_unique_patterns)}")
print(f"Pattern shape: {perm_matrix.shape}")
print(f"\nFirst 5 permutations:")
for i in range(min(5, len(all_unique_patterns))):
    print(f"  Perm {i+1}: {all_unique_patterns[i]}")